In [1]:
%%bash

wget "https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/pub.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/priv.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/model.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/task_template.py" -q

In [2]:
import os
import sys
import torch
import pandas as pd
import requests
import random
import argparse

from pathlib import Path
from torch.utils.data import Dataset
from torchvision.models import resnet18
import torchvision.transforms as transforms

In [3]:
# config for cluster file
# BASE = Path(__file__).parent
# PUB_PATH = BASE / "pub.pt"
# PRIV_PATH = BASE / "priv.pt"
# MODEL_PATH = BASE / "model.pt"
# OUTPUT_CSV = BASE / "submission.csv"

# config for online jupyter notebook
BASE = os.path.dirname(os.path.abspath("__file__"))
PUB_PATH = BASE + "/pub.pt"
PRIV_PATH = BASE + "/priv.pt"
MODEL_PATH = BASE + "/model.pt"
OUTPUT_CSV = BASE + "/submission.csv"

BASE_URL = "http://34.63.153.158"   #DONOT CHANGE
API_KEY = "team_IV 5cd96dee6700b068d36678d03d1b1cec"
TASK_ID = "01-mia"  #DONOT CHANGE

In [4]:
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        label = self.labels[index]
        return id_, img, label

    def __len__(self):
        return len(self.ids)

In [5]:
class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]

In [6]:
# load datasets
print("Loading datasets...")
pub_ds = torch.load(PUB_PATH, weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)

Loading datasets...


In [10]:
help(pub_ds)

Help on MembershipDataset in module __main__ object:

class MembershipDataset(TaskDataset)
 |  MembershipDataset(transform=None)
 |
 |  Method resolution order:
 |      MembershipDataset
 |      TaskDataset
 |      torch.utils.data.dataset.Dataset
 |      typing.Generic
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __getitem__(self, index)
 |
 |  __init__(self, transform=None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  ----------------------------------------------------------------------
 |  Data and other attributes defined here:
 |
 |  __annotations__ = {}
 |
 |  __parameters__ = ()
 |
 |  ----------------------------------------------------------------------
 |  Methods inherited from TaskDataset:
 |
 |  __len__(self)
 |
 |  ----------------------------------------------------------------------
 |  Methods inherited from torch.utils.data.dataset.Dataset:
 |
 |  __add__(self, other: 'Dataset[_T_co]') -> 'ConcatDataset[_T_co]'
 |
 |  -

In [20]:
# normalization (same as training)
MEAN = [0.7406, 0.5331, 0.7059]
STD = [0.1491, 0.1864, 0.1301]

transform = transforms.Compose([
    transforms.Resize(32),
    transforms.Normalize(mean=MEAN, std=STD),
])

pub_ds.transform = transform
priv_ds.transform = transform

In [21]:
# load model
print("Loading model...")
model = resnet18(weights=None)
model.conv1 = torch.nn.Conv2d(3, 64, 3, 1, 1, bias=False)
model.maxpool = torch.nn.Identity()
model.fc = torch.nn.Linear(512, 9)

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

Loading model...


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [22]:
# create random submission (remove this later or it will rewrite your actual submission)
print("Creating random submission...")
ids = [str(i) for i in priv_ds.ids]

df = pd.DataFrame({
    "id": ids,
    "score": [random.random() for _ in ids]
})

df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

Creating random submission...
Saved: /kaggle/working/submission.csv


In [ ]:
# # submit
# def die(msg):
#     print(msg, file=sys.stderr)
#     sys.exit(1)

# parser = argparse.ArgumentParser(description="Submit a CSV file to the server.")
# args = parser.parse_args()

# submit_path = OUTPUT_CSV

# if not submit_path.exists():
#     die(f"File not found: {submit_path}")

# try:
#     with open(submit_path, "rb") as f:
#         resp = requests.post(
#             f"{BASE_URL}/submit/{TASK_ID}",
#             headers={"X-API-Key": API_KEY},
#             files={"file": (submit_path.name, f, "application/csv")},
#             timeout=(10, 600),
#         )
#     try:
#         body = resp.json()
#     except Exception:
#         body = {"raw_text": resp.text}

#     if resp.status_code == 413:
#         die("Upload rejected: file too large (HTTP 413).")

#     resp.raise_for_status()

#     print("Successfully submitted.")
#     print("Server response:", body)
#     submission_id = body.get("submission_id")
#     if submission_id:
#         print(f"Submission ID: {submission_id}")

# except requests.exceptions.RequestException as e:
#     detail = getattr(e, "response", None)
#     print(f"Submission error: {e}")
#     if detail is not None:
#         try:
#             print("Server response:", detail.json())
#         except Exception:
#             print("Server response (text):", detail.text)
#     sys.exit(1)